# Aplicación HCDN
---

## 01 Scraping

## 02 Formatear archivos

In [18]:
rm(list = ls())
gc()

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,1019034,54.5,2012708,107.5,NA,2012708,107.5
Vcells,1806633,13.8,8388608,64.0,16384,2673912,20.5


In [19]:
library(data.table)
library(pbapply)
library(tidyverse)
library(lubridate)

In [25]:
# Asumo que estoy parado en la carpeta de legislativas
setwd("/Users/ppando/Materias/cs-datos/especializacion/ciencia-datos/clases/legislativas/votaciones_csvs")

In [29]:
# Listo los archivos cuyo nombre comienza con un digito (en contraposicion a una letra)
archivos_sesiones <- list.files(pattern = "^votacion_\\d{4}")

In [30]:
# Importo los archivos, les agrego la info de la sesion y cambio los nombres de las columnas
sesiones <- pblapply(archivos_sesiones,function(archivo){
  numero_sesion <- str_extract(archivo,"votacion_\\d{4}")
  numero_sesion <- str_extract(numero_sesion,"\\d{4}")
  sesion <- fread(archivo)
  sesion$id_votacion <- as.integer(numero_sesion)
  setnames(sesion
           ,c("DIPUTADO","BLOQUE","PROVINCIA","¿CÓMO VOTÓ?")
           ,c("legislador","bloque","provincia","voto")
           )
  sesion
})

In [31]:
# Levanto la tabla escrapeada con informacion de cada sesion
archivo_info_sesiones <- list.files(pattern = "votaciones_hcdn_main")
info_sesiones <- fread(archivo_info_sesiones)

info_sesiones[,fecha_hora:=dmy_hm(fecha)]
info_sesiones[,yr:=year(fecha_hora)]

sesiones_dt <- rbindlist(sesiones)

sesiones_dt[info_sesiones,yr:=i.yr,on=.(id_votacion)]

In [32]:
# Legisladores
sesiones_dt[,legislador_norm:=stringi::stri_trans_general(tolower(legislador),"Latin-ASCII")]
setorder(sesiones_dt,legislador_norm)
sesiones_dt[,legislador_id:=.GRP,.(legislador_norm)]
sesiones_dt[,legislador_norm:=NULL]

In [33]:
# Bloques
sesiones_dt[,bloque_norm:=stringi::stri_trans_general(tolower(bloque),"Latin-ASCII")]
setorder(sesiones_dt,bloque_norm)
sesiones_dt[,bloque_id:=.GRP,.(bloque_norm)]
sesiones_dt[,bloque_norm:=NULL]

In [ ]:
# Provincias
sesiones_dt[,provincia_norm:=stringi::stri_trans_general(tolower(provincia),"Latin-ASCII")]
setorder(sesiones_dt,provincia_norm)
sesiones_dt[,provincia_id:=.GRP,.(provincia_norm)]
sesiones_dt[,provincia_norm:=NULL]

In [34]:
# Guardo las tablas
setwd("../")
dir.create("sesiones",showWarnings = FALSE)
setwd("sesiones")
fwrite(info_sesiones,"info_sesiones_dt.csv")
fwrite(sesiones_dt,"sesiones_dt.csv.gz")

In [36]:
setwd("/Users/ppando/Materias/cs-datos/especializacion/ciencia-datos/clases/legislativas")

## 03 Generar ideal points

In [37]:
rm(list = ls())
gc()

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,1078028,57.6,2012708,107.5,NA,2012708,107.5
Vcells,2766902,21.2,10457981,79.8,16384,10457981,79.8


In [40]:
library(data.table)
library(pbapply)
library(tidyverse)
library(lubridate)
library(pscl)

In [41]:
# Asumo que estoy parado en la carpeta de legislativas
carpeta_legislativas <- "/Users/ppando/Materias/cs-datos/especializacion/ciencia-datos/clases/legislativas"
setwd(carpeta_legislativas)
setwd("./sesiones")
sesiones <- fread("sesiones_dt.csv.gz")

anios <- sort(unique(sesiones$yr))

sesiones <- sesiones[voto %in% c("AUSENTE","AFIRMATIVO","NEGATIVO","ABSTENCION")]

setorder(sesiones,yr,id_votacion)
sesiones[,id:=seq(1,nrow(sesiones))]

gc()

,used,(Mb),gc trigger,(Mb),limit (Mb),max used,(Mb)
Ncells,1130819,60.4,2012708,107.5,NA,2012708,107.5
Vcells,2701200,20.7,10457981,79.8,16384,10457981,79.8


In [42]:
idps <- lapply(anios,function(x){
  
  votos_anio <- sesiones[yr==x]
  nvotos_periodo <- dim(votos_anio)[1]
  legis.names <- unique(votos_anio$legislador)
  nlegis <- length(legis.names) # Cantidad de legisladores
  legis.partidos <- unique(votos_anio[,.(bloque=last(bloque)),legislador])$bloque
  actas.names <- unique(votos_anio[,.(id_votacion)])$id_votacion # Ids de las actas votadas
  nactas <- length(actas.names) # Cantidad de actas votadas
  niveles <- unique(votos_anio[,voto]) # Variantes de voto ("AFIRMATIVO", "NEGATIVO", etc.)
  niveles <- sort(niveles)
  
  notInLegis <- match('AUSENTE', niveles)
  yea <- match('AFIRMATIVO', niveles)
  nay <- match('NEGATIVO', niveles)
  abstencion <- match('ABSTENCION', niveles)
  
  # Genero matriz vacia con dimension nlegis x nactas
  data <- matrix(, nlegis, nactas)
  
  # Matriz con votos por legislador y acta con valores numericos de acuerdo al 'nivel'
  for (n in 1:nvotos_periodo) {
    nombre <- legis.names == votos_anio[n, legislador]
    acta <- actas.names == votos_anio[n, id_votacion]
    nivel <- match(votos_anio[n, voto], niveles)
    data[nombre, acta] <- nivel
  }
  
  legis.data <- data.frame(party=legis.partidos) # Convierto la tabla con partidos x legislador en formato dataframe de R
  row.names(legis.data) <- legis.names # Asigno los nombres de los legisladores como etiquetas de fila
  desc <- paste('Votos de diputados', x) # Etiqueta descriptiva para el periodo
  # La funcion a continuacion, es especifica de pscl y es esencial para el calculo
  # Se utilizan todos los valores que fueron calculados previamente
  rData <- rollcall(data, yea=yea, nay=nay, missing=abstencion,
                    notInLegis=notInLegis, legis.names=legis.names, vote.names=actas.names,
                    legis.data=legis.data, vote.data=NULL, desc=desc, source=NULL)
  
  # El algoritmo de ideal points en si, en este caso, con valores normalizados
  fitted <- ideal(rData, normalize=T,impute = T)
  outcome <- fitted$xbar # Extraigo el vector de puntos ideales
  
  outcome <- cbind(outcome,anio=x) # Le pego la columna que indica el periodo
  
  outcome <- as.data.frame(outcome) # Lo transformo al formato dataframe de R
  names(outcome) <- c("idp","anio") # Agrego nombres de columna
  outcome <- add_rownames(outcome,"nombre") # Extraigo las etiquetas de fila y las transformo en nueva columna con el nombre de cada legislador
  outcome <- as.data.table(outcome)
})

idps <- rbindlist(idps)

setwd(carpeta_legislativas)

setwd("./diputados/02_puntos ideales")
dir.create("idps",showWarnings = FALSE)
setwd("idps")

fwrite(idps,"idps.csv")

# rm(list = setdiff(ls(),c("idps","anios")))

distancias <- pblapply(anios,function(a){
  idps_anio <- idps[anio==a]
  comb_idps_anio <- as.data.table(expand.grid(nombre1=idps_anio$nombre,nombre2=idps_anio$nombre))
  comb_idps_anio <- comb_idps_anio[idps_anio,on=c("nombre1"="nombre")]
  setnames(comb_idps_anio,"idp","idp_desde")
  comb_idps_anio <- comb_idps_anio[idps_anio[,!"anio"],on=c("nombre2"="nombre")]
  setnames(comb_idps_anio,"idp","idp_hasta")
  comb_idps_anio[,distancia:=abs(idp_desde-idp_hasta)]
})

distancias <- rbindlist(distancias)
distancias <- distancias[distancia>0]

fwrite(distancias,"distancias_idps.csv.gz")

gc()

idps %>% 
  arrange(anio, idp) %>% 
  group_by(anio) %>% 
  mutate(i=1,i=cumsum(i)) %>% 
  ungroup() %>% 
  ggplot(aes(i,y=idp,col=factor(anio))) +
  geom_line()

idps[,.N,by="anio"] %>% 
  ggplot(aes(anio,N)) +
  geom_point()

# sesiones_totales <- readRDS("sesiones_totales.rds")

setorder(sesiones,yr,id_votacion)
legisladores_bloque_provincia <- sesiones[,.(bloque=last(bloque),provincia=last(provincia)),.(legislador,yr)]

fwrite(legisladores_bloque_provincia,"legisladores_bloque_provincia.csv")

ideal: analysis of roll call data via Markov chain Monte Carlo methods.

Ideal Point Estimation

Number of Legislators		 259 
Number of Items			 70 


Starting MCMC Iterations...



Warning message:
“`add_rownames()` was deprecated in dplyr 1.0.0.
ℹ Please use `tibble::rownames_to_column()` instead.”


ideal: analysis of roll call data via Markov chain Monte Carlo methods.

Ideal Point Estimation

Number of Legislators		 265 
Number of Items			 38 


Starting MCMC Iterations...



Warning message:
“`add_rownames()` was deprecated in dplyr 1.0.0.
ℹ Please use `tibble::rownames_to_column()` instead.”


ideal: analysis of roll call data via Markov chain Monte Carlo methods.

Ideal Point Estimation

Number of Legislators		 261 
Number of Items			 170 


Starting MCMC Iterations...



Warning message:
“`add_rownames()` was deprecated in dplyr 1.0.0.
ℹ Please use `tibble::rownames_to_column()` instead.”


ideal: analysis of roll call data via Markov chain Monte Carlo methods.

Ideal Point Estimation

Number of Legislators		 262 
Number of Items			 78 


Starting MCMC Iterations...



Warning message:
“`add_rownames()` was deprecated in dplyr 1.0.0.
ℹ Please use `tibble::rownames_to_column()` instead.”


ERROR: Error in setwd("./diputados/02_puntos ideales"): no es posible cambiar el directorio de trabajo
